## Install `holidays`

The Databricks Serverless environment does not ship the `holidays` package. This cell installs it and restarts Python so `src.features` can import it. Only takes ~5 seconds.


In [0]:
%pip install holidays -q
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 06 — Live API ingestion (AviationStack)

Fetches a live flight, validates against the Silver contract, and writes
`api_bronze_flights` and `api_silver_flights`. Falls back to a committed
fixture when `USE_FIXTURE=True`, so the pipeline stays demoable when the
key is missing or the 100-request free quota is spent.

In [0]:
import sys
sys.path.append("..")

import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

from src import config
from src.api_pipeline import (
    LookupParams, fetch_flights, load_fixture, project_to_silver, validate_schema,
)

## Runtime switches — one flight, plus its alternatives

The previous shape asked the API for every departure from an airport and scored all of
them. That answers a question nobody asked. The product question is narrower and more
useful: **will this flight be late, and if so what else could I take?**

So the ingestion has a subject. `FLIGHT_IATA` names the flight of interest;
`DEP_IATA`/`ARR_IATA` bound the candidate pool it will be compared against. Everything
downstream scores that pool and nothing else.

**On `airline_iata`.** A previous run passed `airline_iata="DL1682"`, which is why it
returned nothing: that field wants a carrier code (`DL`), not a flight number. A flight
number goes in `flight_iata`. The widgets below make that distinction explicit rather than
leaving it to be rediscovered.


In [ ]:
dbutils.widgets.dropdown("USE_FIXTURE", "false", ["true", "false"])
dbutils.widgets.text("FLIGHT_IATA", "DL1682")   # the flight of interest, e.g. DL1682
dbutils.widgets.text("DEP_IATA", "ATL")         # route origin, for the candidate pool
dbutils.widgets.text("ARR_IATA", "IAH")         # route destination
dbutils.widgets.dropdown("DROP_CODESHARES", "true", ["true", "false"])

USE_FIXTURE = dbutils.widgets.get("USE_FIXTURE").lower() == "true"
FLIGHT_OF_INTEREST = (dbutils.widgets.get("FLIGHT_IATA") or "").strip().upper() or None
DEP_IATA = (dbutils.widgets.get("DEP_IATA") or "").strip().upper() or None
ARR_IATA = (dbutils.widgets.get("ARR_IATA") or "").strip().upper() or None
DROP_CODESHARES = dbutils.widgets.get("DROP_CODESHARES") == "true"

print(f"Flight of interest : {FLIGHT_OF_INTEREST}")
print(f"Candidate pool     : {DEP_IATA} -> {ARR_IATA}")
print(f"Drop codeshares    : {DROP_CODESHARES}")


## Fetch

In [ ]:
def _rows(payload):
    return payload.get("data") or []


def _try(label, params, access_key):
    """Run one query shape and report what it returned."""
    try:
        payload = fetch_flights(params, access_key)
        n = len(_rows(payload))
        total = (payload.get("pagination") or {}).get("total")
        print(f"  {label:<34} {n:>4} records   total={total}")
        return payload
    except Exception as e:
        print(f"  {label:<34} failed: {type(e).__name__}: {e}")
        return {"data": []}


if USE_FIXTURE:
    payload = load_fixture()
    print(f"Using fixture ({len(_rows(payload))} records)")
else:
    access_key = dbutils.secrets.get(
        scope=config.AVIATIONSTACK_SECRET_SCOPE,
        key=config.AVIATIONSTACK_SECRET_KEY,
    )

    # Two queries, each with a job. The flight of interest is looked up by flight
    # number; the candidate pool is the route it runs on. Route first, because it
    # usually already contains the flight of interest.
    print("Fetching:")
    route = _try(
        f"route {DEP_IATA}->{ARR_IATA}",
        LookupParams(dep_iata=DEP_IATA, arr_iata=ARR_IATA, limit=100),
        access_key,
    )
    records = list(_rows(route))

    if not records and DEP_IATA:
        # A specific route can legitimately be empty in the provider's snapshot.
        route = _try(f"origin {DEP_IATA} only",
                     LookupParams(dep_iata=DEP_IATA, limit=100), access_key)
        records = list(_rows(route))

    have = {(r.get("flight") or {}).get("iata") for r in records}
    if FLIGHT_OF_INTEREST and FLIGHT_OF_INTEREST not in have:
        focus = _try(f"flight {FLIGHT_OF_INTEREST}",
                     LookupParams(flight_iata=FLIGHT_OF_INTEREST, limit=100), access_key)
        records = _rows(focus) + records

    payload = {"data": records, "pagination": {"count": len(records)}}
    print(f"\nCombined: {len(records)} records")

    if not records:
        print("\nEverything came back empty. That points at the account or plan rather")
        print("than the filter — check the request quota and whether the plan covers")
        print("the /flights endpoint. Re-run with USE_FIXTURE=true to exercise the rest.")


## Write API Bronze (raw JSON payload as string)

In [0]:
import json as _json

raw_row = [(_json.dumps(payload),)]
bronze_df = (
    spark.createDataFrame(raw_row, ["raw_payload"])
    .withColumn("ingested_at", current_timestamp())
    .withColumn("used_fixture", lit(USE_FIXTURE))
)
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.API_BRONZE)
)
print(f"Appended to {config.API_BRONZE}")

Appended to workspace.flights.api_bronze_flights


## Project to Silver + DQ check

In [ ]:
silver_pdf = project_to_silver(payload)
report = validate_schema(silver_pdf)
print(report)

HAS_ROWS = len(silver_pdf) > 0

if HAS_ROWS:
    before = len(silver_pdf)

    # Codeshares are the same aircraft sold under another carrier's number. Keeping
    # them inflates the candidate pool and lets the recommender offer a flight as an
    # alternative to itself: ATL-LAX 18:47 arrived as DL753, WS6993 and AM4626.
    if DROP_CODESHARES and "is_codeshare" in silver_pdf.columns:
        marketing = silver_pdf[silver_pdf["is_codeshare"]]
        silver_pdf = silver_pdf[~silver_pdf["is_codeshare"]].copy()
        print(f"\nDropped {len(marketing)} codeshare rows "
              f"({before} -> {len(silver_pdf)} operating flights)")

    # Belt and braces: two operating carriers genuinely scheduled at the same
    # minute on the same route is vanishingly rare, and almost always a codeshare
    # the flag missed.
    dupe_key = ["origin_airport_code", "destination_airport_code",
                "flight_date", "crs_dep_time"]
    dupes = silver_pdf.duplicated(subset=dupe_key, keep="first").sum()
    if dupes:
        silver_pdf = silver_pdf.drop_duplicates(subset=dupe_key, keep="first").copy()
        print(f"Collapsed {dupes} further same-minute duplicates on the same route")

    silver_pdf["is_flight_of_interest"] = (
        silver_pdf["flight_iata"].fillna("").str.upper() == (FLIGHT_OF_INTEREST or "")
    )
    n_focus = int(silver_pdf["is_flight_of_interest"].sum())

    print(f"\nCandidate pool: {len(silver_pdf)} operating flights")
    if FLIGHT_OF_INTEREST:
        if n_focus:
            row = silver_pdf[silver_pdf["is_flight_of_interest"]].iloc[0]
            print(f"Flight of interest {FLIGHT_OF_INTEREST}: found "
                  f"({row['origin_airport_code']}->{row['destination_airport_code']} "
                  f"on {row['flight_date']})")
        else:
            print(f"Flight of interest {FLIGHT_OF_INTEREST}: NOT in the pool — the "
                  f"route's flights will still be scored as alternatives.")
    HAS_ROWS = len(silver_pdf) > 0
else:
    print("\nNo flights projected. Skipping the Silver write; the OpenSky cells below")
    print("still run, so the live layer stays verifiable on its own.")


---

## Live aircraft state (OpenSky) — one call for the whole airspace

AviationStack is queried per origin/destination pair. With 7,675 distinct routes in the
historical data, covering them once costs 7,675 calls, and refreshing costs that again every
cycle. That does not improve with a paid tier; the shape is wrong.

OpenSky's `/states/all` returns every aircraft the network is tracking in a single request.
Measured against the continental US box: **8,166 aircraft in one 1.1 MB response**, 2,893 of
them carrying callsigns belonging to carriers present in the BTS data. Call volume stops
scaling with the number of flights and scales only with polling frequency.

**The division of labour.** OpenSky decides *which* flights are worth asking about;
AviationStack answers *how late* they were.

That split is not arbitrary. The model trains on BTS `DEP_DELAY`, which is **gate** departure
delay. OpenSky observes when an airframe stops being `on_ground`, which is **wheels-off**.
They differ by taxi-out — minutes at a small field, far more at a congested hub — so deriving
delay from OpenSky would inject a bias that is worst precisely where delay matters most, and
would do it silently. AviationStack's `departure.delay` is already gate semantics, so that is
the value served. `docs/API_STRATEGY.md` has the full argument.

**The join needs no mapping table.** AviationStack returns `flight.icao` (`"DAL1234"`);
OpenSky broadcasts `callsign` in the same ICAO form, space-padded. After stripping they
compare directly, because AviationStack supplies the ICAO spelling itself.

**What `on_ground` buys.** It is the phase split the two models need, arriving free with data
already fetched. `True` is the pre-departure population, `False` is in-flight. Without it both
models score every flight regardless of whether it has left the gate, which is incoherent.


In [ ]:
from src.opensky import (
    CONUS_BBOX, OpenSkyClient, match_to_schedule, parse_states, split_by_phase,
)

dbutils.widgets.dropdown("USE_OPENSKY", "true", ["true", "false"])
USE_OPENSKY = dbutils.widgets.get("USE_OPENSKY") == "true"

state_rows, phase_by_icao, opensky_report = [], {}, None

if not USE_OPENSKY:
    print("USE_OPENSKY=false — skipping the live state feed.")
else:
    try:
        # The client id and secret are long-lived and live in Databricks secrets.
        # The bearer token minted from them lasts 1,800 seconds, so the client
        # refreshes on demand rather than the notebook handling tokens at all.
        # Nothing here ever needs re-saving to the secret scope.
        opensky = OpenSkyClient(
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_ID_KEY),
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_SECRET_KEY),
        )

        payload_os = opensky.fetch_states(CONUS_BBOX)
        state_rows = parse_states(payload_os)
        ground, airborne = split_by_phase(state_rows)

        print(f"OpenSky: {len(state_rows):,} aircraft in ONE call "
              f"(token refreshes: {opensky.refresh_count})")
        print(f"  on ground : {len(ground):,}   (pre-departure population)")
        print(f"  airborne  : {len(airborne):,}   (in-flight population)")

        # Match on the flight designator only, deliberately ignoring the date.
        #
        # Every match failed on the previous run because AviationStack's plan
        # returned flights dated 2026-08-31 while the OpenSky snapshot was live on
        # 2026-09-14. A flight from a fortnight ago is not in the air now, so
        # matching a dated flight against a live snapshot can only ever return zero.
        #
        # A flight number is a recurring daily service, though. "Is DL1682 airborne
        # right now?" is answerable and useful even when the schedule record on hand
        # describes an earlier operating day — and it is the question the phase split
        # actually needs answered. The date mismatch is reported rather than hidden.
        scheduled_icao = [c for c in silver_pdf.get("flight_icao", []) if c]
        matched, opensky_report = match_to_schedule(state_rows, scheduled_icao)

        print(f"\nMatched against {opensky_report['scheduled_flights']} scheduled flights:")
        for k, v in opensky_report.items():
            shown = f"{v:.1%}" if isinstance(v, float) else v
            print(f"  {k:<26} {shown}")

        if HAS_ROWS and "flight_date" in silver_pdf.columns and len(silver_pdf):
            import datetime as _dt
            newest = max(silver_pdf["flight_date"])
            age = (_dt.date.today() - newest).days
            if age > 1:
                print(f"\n  NOTE: the newest scheduled flight is {newest}, {age} days old.")
                print( "  The plan in use is serving historical rather than live schedules.")
                print( "  Matching is by flight number, so a recurring service can still be")
                print( "  located in today's snapshot, but any match describes today's")
                print( "  operation of that number and not the dated record it came from.")

        phase_by_icao = {
            r["callsign"]: ("airborne" if r["on_ground"] is not True else "on_ground")
            for r in matched
        }

        print("\nA low match rate is expected and is not by itself a defect. Around 39% of")
        print("aircraft aloft over the US are N-registered general aviation that will never")
        print("appear in a BTS schedule, and a scheduled flight only appears in the snapshot")
        print("if it happens to be moving right now — most of a day's schedule is not.")
        print("Measure the rate, publish it, do not assume it.")
    except Exception as e:
        # The live feed is an enhancement, not a dependency. Bronze -> Gold ->
        # train never touches an API, and scoring falls back to pre-departure.
        print(f"OpenSky unavailable ({type(e).__name__}: {e}).")
        print("Continuing without the phase split — every flight falls back to pre-departure.")


### Landing the snapshot and attaching phase

The raw snapshot goes to its own Delta table so a scoring run can be reconstructed later —
what the airspace looked like at the moment a prediction was made is exactly the evidence you
need to audit that prediction afterwards.

`flight_phase` carries three values, and the third is the honest one:

- `airborne` — OpenSky sees it flying, so it has departed and `dep_delay` is real
- `on_ground` — OpenSky sees it, still at the field
- `unknown` — not matched. No callsign, outside the box, or general aviation. These fall back
  to the pre-departure model, which is the safe default: it is the variant that does not
  require a departure to have happened.


In [0]:
if state_rows:
    states_sdf = (
        spark.createDataFrame(pd.DataFrame(state_rows))
        .withColumn("ingested_at", current_timestamp())
    )
    (
        states_sdf.write.format("delta").mode("append")
        .option("mergeSchema", "true").saveAsTable(config.OPENSKY_STATES)
    )
    print(f"Appended {len(state_rows):,} state vectors -> {config.OPENSKY_STATES}")

if not HAS_ROWS:
    print("\nNo scheduled flights to attach phase to. The OpenSky snapshot above is")
    print("still written, so the live feed is verifiable on its own.")
else:
    # Attach phase, THEN write. Writing first left flight_phase on a frame nothing
    # read afterwards, and 07_score reported the column missing every run.
    silver_pdf["flight_phase"] = (
        silver_pdf["flight_icao"].map(phase_by_icao).fillna("unknown")
        if "flight_icao" in silver_pdf.columns and phase_by_icao
        else "unknown"
    )
    counts = silver_pdf["flight_phase"].value_counts().to_dict()
    print(f"\nPhase assignment across {len(silver_pdf)} scheduled flights: {counts}")

    silver_sdf = (
        spark.createDataFrame(silver_pdf)
        .withColumn("ingested_at", current_timestamp())
    )
    (
        silver_sdf.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(config.API_SILVER)
    )
    print(f"Appended {silver_sdf.count()} rows to {config.API_SILVER} (with flight_phase)")

    has_delay = silver_pdf["dep_delay"].notna().sum()
    print(f"\nRows carrying an AviationStack gate dep_delay: {has_delay} of {len(silver_pdf)}")
    print("Those are the rows the in-flight model can score. The rest get the")
    print("pre-departure model, which is the only one that works before pushback.")


## Log DQ result to the data-quality table

In [0]:
from pyspark.sql import Row

dq_row = spark.createDataFrame([
    Row(
        checked_at=report["checked_at"],
        source="aviationstack",
        used_fixture=USE_FIXTURE,
        row_count=report["row_count"],
        passed=report["passed"],
        missing_columns=",".join(report["missing_columns"]),
    )
])
(
    dq_row.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.DATA_QUALITY_LOG)
)

### What OpenSky can see on its own

Independent of the schedule feed. If AviationStack returns nothing, this cell still shows
whether the live layer is healthy — and it separates "the API is broken" from "that query was
too narrow", which look identical from a single empty result.

It also demonstrates the capability the phase split rests on: aircraft sitting at a gate and
aircraft in the air are both visible, per airport, from one call each.


In [0]:
if not state_rows:
    print("No OpenSky snapshot to probe — the earlier cell did not return states.")
else:
    from src.opensky import CONUS_BBOX

    # Hub coordinates with a ~0.25 degree box, about 25 km on a side.
    HUBS = {
        "ATL": (33.6407, -84.4277), "LAX": (33.9416, -118.4085),
        "ORD": (41.9742, -87.9073), "DFW": (32.8998, -97.0403),
        "DEN": (39.8561, -104.6737), "JFK": (40.6413, -73.7781),
    }
    PAD = 0.25

    print(f"{'airport':<9}{'total':>8}{'on ground':>11}{'airborne':>10}  sample ground callsigns")
    print("-" * 82)
    for code, (lat, lon) in HUBS.items():
        try:
            box = {"lamin": lat - PAD, "lamax": lat + PAD,
                   "lomin": lon - PAD, "lomax": lon + PAD}
            rows = parse_states(opensky.fetch_states(box))
            g, a = split_by_phase(rows)
            sample = ", ".join(r["callsign"] for r in g if r["callsign"])
            print(f"{code:<9}{len(rows):>8}{len(g):>11}{len(a):>10}  {sample[:44] or '(none)'}")
        except Exception as e:
            print(f"{code:<9}  probe failed: {type(e).__name__}")

    print("-" * 82)
    print("Both phases are visible at every hub, from one call per airport. That is")
    print("the capability the two-model split rests on: `on_ground` separates the")
    print("flights the pre-departure model should answer for from the ones the")
    print("in-flight model can.")
    print(f"\nToken refreshes so far this run: {opensky.refresh_count}")
